In [ ]:
import os
import numpy as np
import re  # 新增正则模块

# 新增数字提取函数
def extract_number(filename):
    """从文件名中提取末尾的数字（支持任意前缀）"""
    match = re.search(r'_(\d+)\.txt$', filename)
    return int(match.group(1)) if match else -1

data_dict = {
    "c": [],        # c_*.txt
    "p": [],        # p_*.txt
    "v1": [],        # v_*.txt
    "bdry_points_0": [],  # bdry_points_*.txt
    "ctrl_points_0": [],  # ctrl_points_*.txt
    "ctrl_0_nxny": [],
    "bdry_points_1": [],  # bdry_points_*.txt
    "ctrl_points_1": [],  # ctrl_points_*.txt
    "ctrl_1_nxny": []
}

base_dirs = ["solutions", "points"]

for base_dir in base_dirs:
    if not os.path.exists(base_dir):
        continue
    
    # 修改排序方式：按数字大小排序
    for filename in sorted(os.listdir(base_dir), key=extract_number):  # 关键修改
        filepath = os.path.join(base_dir, filename)
        
        if filename.startswith("c_"):
            data_dict["c"].append(np.loadtxt(filepath))
        elif filename.startswith("p_"):
            data_dict["p"].append(np.loadtxt(filepath))
        elif filename.startswith("v_"):
            data_dict["v1"].append(np.loadtxt(filepath))
        elif filename.startswith("bdry_points_0_"):
            data_dict["bdry_points_0"].append(np.loadtxt(filepath))
        elif filename.startswith("ctrl_points_0_"):
            data_dict["ctrl_points_0"].append(np.loadtxt(filepath))
        elif filename.startswith("ctrl_0_nxny_"):
            data_dict["ctrl_0_nxny"].append(np.loadtxt(filepath))
        elif filename.startswith("bdry_points_1_"):
            data_dict["bdry_points_1"].append(np.loadtxt(filepath))
        elif filename.startswith("ctrl_points_1_"):
            data_dict["ctrl_points_1"].append(np.loadtxt(filepath))
        elif filename.startswith("ctrl_1_nxny_"):
            data_dict["ctrl_1_nxny"].append(np.loadtxt(filepath))

c_list = [np.array(arr) for arr in data_dict["c"]]
p_list = [np.array(arr) for arr in data_dict["p"]]
v1_list = [np.array(arr) for arr in data_dict["v1"]]
bdry_points_0_list = [np.array(arr) for arr in data_dict["bdry_points_0"]]
ctrl_points_0_list = [np.array(arr) for arr in data_dict["ctrl_points_0"]]
ctrl_0_nxny_list = [np.array(arr) for arr in data_dict["ctrl_0_nxny"]]
bdry_points_1_list = [np.array(arr) for arr in data_dict["bdry_points_1"]]
ctrl_points_1_list = [np.array(arr) for arr in data_dict["ctrl_points_1"]]
ctrl_1_nxny_list = [np.array(arr) for arr in data_dict["ctrl_1_nxny"]]

print(f"c_list: {len(c_list)}, p_list: {len(p_list)}, v_list: {len(v1_list)}, bdry_points_0_list: {len(bdry_points_0_list)}, ctrl_points_0_list: {len(ctrl_points_0_list)}, ctrl_0_nxny_list: {len(ctrl_0_nxny_list)}, bdry_points_1_list: {len(bdry_points_1_list)}, ctrl_points_1_list: {len(ctrl_points_1_list)}, ctrl_1_nxny_list: {len(ctrl_1_nxny_list)}")

In [ ]:
steps = max(len(c_list), len(p_list), len(v1_list), len(bdry_points_0_list), len(ctrl_points_0_list), len(ctrl_0_nxny_list),len(bdry_points_1_list), len(ctrl_points_1_list), len(ctrl_1_nxny_list))

In [ ]:
x_min = -5
y_min = -5
x_max = 5
y_max = 5

In [ ]:
def circularity_from_contour(x, y):
    """
    计算由离散点 (x, y) 构成的闭合轮廓的圆度（circularity）。
    
    圆度 = 4 * π * Area / Perimeter²
    - 理想圆：1.0
    - 越偏离圆形：越小（>0）
    
    参数：
        x, y : array-like, shape (N,)
            轮廓点的 x 和 y 坐标（按顺序，建议闭合：首尾点可相同或不同）
    
    返回：
        float: 圆度值（0 < circularity ≤ 1）
    """
    x = np.asarray(x)
    y = np.asarray(y)
    
    if len(x) < 3:
        raise ValueError("At least 3 points are required to form a contour.")
    
    # 确保首尾相连（便于计算 perimeter 和 area）
    if not (np.isclose(x[0], x[-1]) and np.isclose(y[0], y[-1])):
        x = np.append(x, x[0])
        y = np.append(y, y[0])
    
    # 计算周长（折线段长度和）
    dx = np.diff(x)
    dy = np.diff(y)
    perimeter = np.sum(np.hypot(dx, dy))
    
    # 计算面积（Shoelace 公式）
    # Area = 0.5 * |Σ(x_i * y_{i+1} - x_{i+1} * y_i)|
    area = 0.5 * np.abs(np.dot(x[:-1], y[1:]) - np.dot(x[1:], y[:-1]))
    
    if perimeter == 0:
        return 0.0
    
    circularity = (4 * np.pi * area) / (perimeter ** 2)
    return circularity

In [ ]:
from scipy.interpolate import splprep, splev
import matplotlib.pyplot as plt

def plot_control_points_steps(ctrl_points0_list, ctrl_points1_list, record_t, steps_to_plot):
    # ====== 【新增】准备存储圆度 ======
    all_times = []
    all_circ_inner = []
    all_circ_outer = []
    # ===================================

    plt.figure(figsize=(8, 8), dpi=600)
    colors = plt.cm.plasma(np.linspace(0, 1, len(steps_to_plot)))

    for idx, t in enumerate(steps_to_plot):
        ctrl_points0 = ctrl_points0_list[t]
        ctrl_points1 = ctrl_points1_list[t]

        circ0 = np.nan  # 【新增】初始化
        circ1 = np.nan  # 【新增】

        # -------- 内边界（0） --------
        if ctrl_points0 is None or len(ctrl_points0) == 0:
            print(f"⚠️ Step {t}: inner boundary empty — skip inner plotting")
        else:
            ctrl_points0 = np.asarray(ctrl_points0)
            n0 = ctrl_points0.shape[0]

            if n0 >= 3:
                closed_points0 = np.vstack([ctrl_points0, ctrl_points0[0]])
                try:
                    k0 = min(3, n0 - 1)
                    tck0, _ = splprep([closed_points0[:, 0], closed_points0[:, 1]], s=0, per=True, k=k0)
                    x0_smooth, y0_smooth = splev(np.linspace(0, 1, 400), tck0)
                    plt.plot(x0_smooth, y0_smooth, color=colors[idx], linestyle='--',
                             label=f"Inner t={round(record_t[t], 4)}")
                    # 【新增】用平滑曲线计算圆度（更准）
                    circ0 = circularity_from_contour(x0_smooth, y0_smooth)
                except Exception as e:
                    print(f"⚠️ Step {t}: inner splprep failed ({e}), using polyline")
                    closed_for_calc = np.vstack([ctrl_points0, ctrl_points0[0]])
                    plt.plot(*closed_for_calc.T, color=colors[idx], linestyle='--',
                             label=f"Inner t={round(record_t[t], 4)}")
                    circ0 = circularity_from_contour(closed_for_calc[:, 0], closed_for_calc[:, 1])
            else:
                if n0 == 1:
                    plt.plot(ctrl_points0[:, 0], ctrl_points0[:, 1], 'o', color=colors[idx],
                             label=f"Inner t={round(record_t[t], 4)}")
                    circ0 = 0.0
                else:
                    closed_for_calc = np.vstack([ctrl_points0, ctrl_points0[0]])
                    plt.plot(closed_for_calc[:, 0], closed_for_calc[:, 1],
                             color=colors[idx], linestyle='--', label=f"Inner t={round(record_t[t], 4)}")
                    circ0 = circularity_from_contour(closed_for_calc[:, 0], closed_for_calc[:, 1])
            
            print(f"Inner circularity at t={round(record_t[t], 4)}: {circ0:.5f}")

        # -------- 外边界（1） --------
        if ctrl_points1 is None or len(ctrl_points1) == 0:
            print(f"⚠️ Step {t}: outer boundary empty — skip outer plotting")
        else:
            ctrl_points1 = np.asarray(ctrl_points1)
            n1 = ctrl_points1.shape[0]

            if n1 >= 3:
                closed_points1 = np.vstack([ctrl_points1, ctrl_points1[0]])
                try:
                    k1 = min(3, n1 - 1)
                    tck1, _ = splprep([closed_points1[:, 0], closed_points1[:, 1]], s=0, per=True, k=k1)
                    x1_smooth, y1_smooth = splev(np.linspace(0, 1, 400), tck1)
                    plt.plot(x1_smooth, y1_smooth, color=colors[idx], linestyle='-',
                             label=f"Outside t={round(record_t[t], 4)}")
                    circ1 = circularity_from_contour(x1_smooth, y1_smooth)
                except Exception as e:
                    print(f"⚠️ Step {t}: outer splprep failed ({e}), using polyline")
                    closed_for_calc = np.vstack([ctrl_points1, ctrl_points1[0]])
                    plt.plot(*closed_for_calc.T, color=colors[idx], linestyle='-',
                             label=f"Outside t={round(record_t[t], 4)}")
                    circ1 = circularity_from_contour(closed_for_calc[:, 0], closed_for_calc[:, 1])
            else:
                if n1 == 1:
                    plt.plot(ctrl_points1[:, 0], ctrl_points1[:, 1], 'o', color=colors[idx],
                             label=f"Outside t={round(record_t[t], 4)}")
                    circ1 = 0.0
                else:
                    closed_for_calc = np.vstack([ctrl_points1, ctrl_points1[0]])
                    plt.plot(closed_for_calc[:, 0], closed_for_calc[:, 1],
                             color=colors[idx], linestyle='-', label=f"Outside t={round(record_t[t], 4)}")
                    circ1 = circularity_from_contour(closed_for_calc[:, 0], closed_for_calc[:, 1])
                
            print(f"Outer circularity at t={round(record_t[t], 4)}: {circ1:.5f}")

        # ====== 【新增】保存当前步的圆度和时间 ======
        all_times.append(record_t[t])
        all_circ_inner.append(circ0)
        all_circ_outer.append(circ1)
        # ============================================

    # --- 原图设置 ---
    plt.xlim([x_min, x_max])
    plt.ylim([y_min, y_max])
    plt.gca().set_aspect("equal")
    plt.grid(True)
    plt.legend(loc='lower right', fontsize=8)
    plt.title("Tumor Inner and Outside Boundaries at Selected Time Steps")
    plt.show()

    # ====== 【新增】画圆度随时间演化图（全新 figure）======
    plt.figure(figsize=(8, 4), dpi=600)
    all_times = np.array(all_times)
    all_circ_inner = np.array(all_circ_inner)
    all_circ_outer = np.array(all_circ_outer)

    # Plot outer (solid), inner (dashed) — consistent with your style
    plt.plot(all_times, all_circ_outer, 'o-', color='tab:blue', label='Outer Boundary')
    plt.plot(all_times, all_circ_inner, 's--', color='tab:orange', label='Inner Boundary')

    plt.xlabel('Time', fontsize=12)
    plt.ylabel('Circularity ($4\\pi A / P^2$)', fontsize=12)
    plt.title('Circularity Evolution of Tumor Boundaries', fontsize=14)
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.legend()
    plt.tight_layout()
    plt.show()
    # ====================================================

In [ ]:
plot_control_points_steps(ctrl_points_0_list, ctrl_points_1_list, np.loadtxt("./time_log.txt"), [0, 7, 12, 17, 22, 27, 32, 37])